# Topic: Model Deployment & Rollout Strategies

## Definition (30-second explanation)
*   *Analogy:* Releasing a new model is like adding a new dish to a restaurant's menu. 
    *   **Shadow:** You cook the dish during dinner rush to see if the kitchen can handle the speed, but you throw it in the trash instead of serving it.
    *   **Canary:** You serve it to just one or two VIP tables to make sure nobody gets food poisoning.
    *   **A/B Testing:** You serve the new dish to half the restaurant and the old dish to the other half, then calculate who left a bigger tip.

## Why Interviewers Ask This
*   Replacing a production model instantly for 100% of users (a "Big Bang" deployment) is an engineering cardinal sin. Interviewers want to ensure you know how to minimize the "blast radius" if your new model hallucinates, crashes, or degrades business metrics.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** How do you test a model safely in the real world without risking brand reputation or causing system outages?
*   **The Mechanism:** An API Gateway or Load Balancer sits in front of your Model Registry. When a user requests a prediction, the router checks a configuration file and directs the traffic (e.g., 90% to the stable `v1` endpoint, 10% to the experimental `v2` endpoint).
*   **The Trade-off:** Running these strategies requires operating multiple models simultaneously in production, which doubles your compute/GPU costs and significantly increases infrastructure complexity.

## When to Use
*   **Shadow Deployment:** Use for major architectural overhauls (e.g., switching from XGBoost to a Deep Learning model) where you need to verify latency and stability without users ever seeing the output.
*   **Canary Rollout:** Use for standard model updates to catch hidden errors. You roll out to 5% of users, monitor error rates for an hour, then scale to 20%, 50%, and 100%.
*   **A/B Testing:** Use when you need to prove business value (e.g., does the new recommendation model actually increase Click-Through Rate?).

## Advantages
*   Zero-downtime deployments (users never see a "System Offline" page).
*   Data-driven decision making rather than guessing if a model is "better."
*   Instant rollback capabilities if a Canary model starts failing.

## Limitations
*   **Shadow:** Cannot measure user reaction or business impact.
*   **A/B Testing:** Can take weeks to achieve statistical significance depending on traffic volume.
*   **Registry Consistency:** If the model registry updates `v2` but the load balancer still thinks `v2` is at an old IP address, traffic gets dropped.

## Common Comparisons
*   **Canary vs. A/B Testing:** This is the most common interview trap! Canary is for **engineering safety** (does it crash? is it slow?). A/B testing is for **business performance** (does it generate more revenue?). 
*   **Blue/Green Deployment vs. Canary:** Blue/Green is a switch (flip 100% of traffic from the old infrastructure to the new infrastructure instantly, keeping the old one as a backup). Canary is a gradual dial (1% -> 10% -> 100%).

## Common Interview Traps
*   **Ignoring "Sticky Sessions":** In A/B testing, if a user is routed to Model B, they must *always* get Model B for the duration of the experiment. If they refresh the page and get Model A, the user experience breaks and your data is ruined.

## Python Syntax
```python
# Conceptual Traffic Router (API Gateway Logic)
import random
import requests

def route_prediction_request(user_features, user_id):
    # 1. Check feature flag / routing configuration
    rollout_config = {"model_v1_weight": 0.90, "model_v2_weight": 0.10}
    
    # 2. Hashing user_id ensures "Sticky Sessions"
    # The same user always gets the same model!
    hash_val = hash(user_id) % 100 
    
    # 3. Route traffic based on the Canary split (10%)
    if hash_val < (rollout_config["model_v2_weight"] * 100):
        # Route to the new Canary model endpoint
        response = requests.post("http://model-v2-cluster/predict", json=user_features)
        model_used = "v2_canary"
    else:
        # Route to the stable Production endpoint
        response = requests.post("http://model-v1-cluster/predict", json=user_features)
        model_used = "v1_stable"
        
    # 4. Log which model served the request for A/B analytics later
    return response.json(), model_used
```

## 45-Second Interview Answer
"To safely deploy a model to production, I rely on gradual rollout strategies rather than 'big bang' releases. For massive architecture changes, I start with a Shadow deployment—running the new model in parallel with the old one to test latency without exposing predictions to the user. For standard updates, I use a Canary rollout, exposing the model to 5% of traffic to monitor for errors before scaling up. If the goal is to measure business impact, like revenue or click-through rate, I implement an A/B test with sticky sessions to statistically prove the new model outperforms the baseline."